# T4 verification: deepfake-autoresearch submission

**What this checks** (the official judges re-run `inference_script.py` on an NVIDIA T4):
1. **Efficiency (20 pts):** wall-clock time for all 3,000 test images. It needs to be **< 5 min** for full marks.
2. **Reproducibility (−5 pt penalty):** do T4 predictions match the `predictions.csv` we generated on the Mac?
3. **Memory:** peak GPU memory (it must not run out of memory).

**Setup (once):**
1. Upload the folder `colab_t4_check/` to the root of **My Drive** (it holds `inference_script.py`, `model_weights.pt`,
   `predictions_mac.csv`, `verify_repro.py`, `requirements.txt`, `test_images.zip`).
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run all cells top to bottom (**Runtime → Run all**), then send back `t4_report.txt` (it's written to the same Drive folder).


In [ ]:
# 1. Confirm we are on a T4
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, sys, platform
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: set Runtime > Change runtime type > T4 GPU"
gpu = torch.cuda.get_device_name(0); print("GPU:", gpu)
if "T4" not in gpu: print("WARNING: not a T4, so timings won't match the judges' hardware")

In [ ]:
# 2. Mount Drive and copy the bundle to local disk (the judges run from local disk; Drive I/O is slow)
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/colab_t4_check'
!ls -la "$BUNDLE"
!rm -rf /content/work && mkdir -p /content/work
!cp "$BUNDLE"/inference_script.py "$BUNDLE"/verify_repro.py "$BUNDLE"/requirements.txt "$BUNDLE"/predictions_mac.csv "$BUNDLE"/model_weights.pt /content/work/
!cd /content/work && unzip -q "$BUNDLE"/test_images.zip -d /content/work && ls /content/work/test | head -3
!echo "test images:" $(ls /content/work/test/images | wc -l)

In [ ]:
# 3. Install deps. Colab already ships a recent torch/torchvision; we only pin timm (the model library).
#    (requirements.txt pins torch 2.14.1 for the official Python 3.10 environment; reinstalling torch here isn't needed.)
!pip -q install timm==1.0.31
import timm, torch, torchvision, PIL, pandas, numpy
print("timm", timm.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)

In [ ]:
# 4. Timed run #1 (cold, the way the judges run it), sampling GPU memory every 0.5 s in the background
import subprocess, time, threading
peak = {"mb": 0}; stop = threading.Event()
def sample():
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
        try: peak["mb"] = max(peak["mb"], int(out.strip().split()[0]))
        except Exception: pass
        time.sleep(0.5)
th = threading.Thread(target=sample, daemon=True); th.start()
t0 = time.time()
r = subprocess.run(["python","inference_script.py","--test_dir","test/images","--weights","model_weights.pt","--output","predictions_t4.csv"],
                   cwd="/content/work", capture_output=True, text=True)
run1 = time.time() - t0; stop.set(); th.join()
print(r.stdout[-1500:]); print(r.stderr[-1500:] if r.returncode else "")
print(f"RUN 1 wall clock: {run1:.1f} s ({run1/60:.2f} min) | peak GPU memory {peak['mb']} MB | exit code {r.returncode}")

In [ ]:
# 5. Timed run #2 (warm) and run-to-run determinism on CUDA
t0 = time.time()
r2 = subprocess.run(["python","inference_script.py","--test_dir","test/images","--weights","model_weights.pt","--output","predictions_t4_run2.csv"],
                    cwd="/content/work", capture_output=True, text=True)
run2 = time.time() - t0
print(r2.stdout[-400:]); print(f"RUN 2 wall clock: {run2:.1f} s")
!cd /content/work && python verify_repro.py predictions_t4.csv predictions_t4_run2.csv

In [ ]:
# 6. Reproducibility: T4 (CUDA, fp16) vs Mac (MPS, fp32), plus format and coverage check
!cd /content/work && python verify_repro.py predictions_t4.csv predictions_mac.csv --test_dir test/images
import pandas as pd, numpy as np
a = pd.read_csv('/content/work/predictions_t4.csv'); b = pd.read_csv('/content/work/predictions_mac.csv')
m = a.merge(b, on='image_id', suffixes=('_t4','_mac'))
flips = int((m.label_t4 != m.label_mac).sum()); dmax = float((m.confidence_t4 - m.confidence_mac).abs().max())
from scipy.stats import spearmanr
rho = spearmanr(m.confidence_t4, m.confidence_mac).correlation
print(f"label flips T4 vs Mac: {flips} | max |dconf| {dmax:.2e} | rank correlation {rho:.6f}")

In [ ]:
# 7. Verdict, saved to Drive as t4_report.txt
eff = 20 if run1 < 300 else 15 if run1 < 600 else 10 if run1 < 1200 else 5 if run1 < 2400 else 0
lines = [
  f"GPU: {gpu} | torch {torch.__version__} | timm {timm.__version__} | python {sys.version.split()[0]}",
  f"Run 1 (cold): {run1:.1f} s = {run1/60:.2f} min -> efficiency band {eff}/20",
  f"Run 2 (warm): {run2:.1f} s",
  f"Peak GPU memory: {peak['mb']} MB",
  f"T4 vs Mac: label flips {flips}, max |dconf| {dmax:.2e}, rank corr {rho:.6f}",
  "RECOMMENDATION: " + ("Mac predictions.csv reproduces on T4. Submit it as-is." if flips == 0 and dmax < 1e-2
      else "Small CUDA/MPS differences. SUBMIT predictions_t4.csv (generated on the judges' hardware) as predictions.csv."),
]
report = "\n".join(lines); print(report)
open(f"{BUNDLE}/t4_report.txt","w").write(report + "\n")
!cp /content/work/predictions_t4.csv "$BUNDLE"/predictions_t4.csv
print("saved t4_report.txt and predictions_t4.csv to", BUNDLE)